# Data Descriptor 

Python object that control how attributes are accessed, set, or deleted on **other objects**.


A descriptor also is an object that implements one or more of these methods: 
- `__get__(self, instance, owner)` - called when accessing the attribute 
- `__set__(self, instance, value)` - called when setting the attribute 
- `__delete__(self, instance)` - called when deleting the attribute

For example,

In [19]:
class PositiveInt:
    def __set_name__(self, owner, name):
        self.private_name = f"_{owner.__name__}__{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        # Get the value from the instance's __dict__
        return instance.__dict__.get(self.private_name)

    def __set__(self, instance, value):
        if not isinstance(value, int) or value <= 0:
            raise ValueError("Value must be a positive integer")
        instance.__dict__[self.private_name] = value


In [20]:
class Person:
    age = PositiveInt()

    def __init__(self, age):
        self.age = age


p = Person(25)
print(p.age)      # 25

p.age = 30        # OK

try:
    p.age = -5        # ValueError
except ValueError: 
    print("The age should be positive")

try:
    p.age = "old"       # ValueError
except ValueError: 
    print("The age should be numeric")

25
The age should be positive
The age should be numeric


## __set_name__ Method

Basically `__set_name__` is a descriptor hook that Python calls automatically at class creation time. It allows a descriptor to learn: 
- Which class own it 
- Under which attribute name it was assigned

For example, 

In [21]:
def __set_name__(self, owner, name):
    self.private_name = "__" + name

In this function,
- `owner` defines the class own it
- `name` defines the attribute it was assigned 

The purpose of this `__set_name__` is that we can use the same descriptor among different attributes without any hard dependency 

In [22]:
class Field:
    def __set_name__(self, owner, name):
        self.private_name = "_" + name

    def __get__(self, instance, owner):
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        setattr(instance, self.private_name, value)

class Person:
    age = Field()
    height = Field()


## __dict__ 

`__dict__` is the actual storage of an object's attribute which mean that 

In [23]:
p.age = 29

is internally stored as 

In [24]:
p.__dict__["_Person__age"] = 29

print(p.__dict__)

{'_Person__age': 29}


## private_name keyword 

It is not a keyword in Python, it is a conventionally chosen string used by a descriptor to decide where **the actual value will be stored inside an instance**

For example,

In [ ]:
def __set_name__(self, owner, name):
    self.private_name = "_" + name

If the descriptor is assigned as: 

In [ ]:
class Person: 
    age = Field()

Then: 

```
self.private_name == "_age"
```

The reason that we need the `private_name`
- The descriptor cannot store data itself
- Instance data must live inside `instance.__dict__`

## getattr()

This method is used for retrieves an attribute dynamically by name

For example, 

```
getattr(p, "age")
```

is equivalent to 

```
p.age
```

## setattr()

This method is used for setting an attribute dynamically 

For example, 

```
setattr(p, "age", 29)
```

is equivalent to 

```
p.age = 29
```